Author: Krish

In [392]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

### User input required
Put the data path on your system in the cell below

In [393]:
data_path = "/Users/viviadams/Downloads/CAR_-_EP_Flow_Activity_Queue__Agent_Names"

### User input ends

### Reading all filenames in the data folder

In [394]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


### Reading all data files
The code chunk below reads and appends all the CAR data files. The first two rows of each file are blank and thus ignored.

In [395]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)

1 CAR - EP, Flow, Activity, Queue, & Agent Names (01-12-25 - 01-18-25) (52010, 7)
2 CAR - EP, Flow, Activity, Queue, & Agent Names (01-19-25 - 02-01-25) (95495, 7)
3 CAR - EP, Flow, Activity, Queue, & Agent Names (02-02-25 - 02-15-25) (90056, 7)
4 CAR - EP, Flow, Activity, Queue, & Agent Names (02-16-25 - 03-01-25) (88186, 7)
5 CAR - EP, Flow, Activity, Queue, & Agent Names (03-02-25 - 03-15-25) (86377, 7)
6 CAR - EP, Flow, Activity, Queue, & Agent Names (04-07-24 - 04-20-24) (88766, 7)
7 CAR - EP, Flow, Activity, Queue, & Agent Names (04-21-24 - 05-04-24) (89643, 7)
8 CAR - EP, Flow, Activity, Queue, & Agent Names (05-05-24 - 05-18-24) (82575, 7)
9 CAR - EP, Flow, Activity, Queue, & Agent Names (05-19-24 - 06-01-24) (71103, 7)
10 CAR - EP, Flow, Activity, Queue, & Agent Names (06-02-24 - 06-15-24) (84354, 7)
11 CAR - EP, Flow, Activity, Queue, & Agent Names (06-16-24 - 06-29-24) (82124, 7)
12 CAR - EP, Flow, Activity, Queue, & Agent Names (06-30-24 - 07-13-24) (79752, 7)
13 CAR - EP, 

### Time datatype conversion
The code chunk below converts time from string to datetime datatype.

In [396]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [397]:
# Checking the datatype of all columns
df_main.dtypes 

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [398]:
# Creating a new column 'hour' as it will be useful to visualize peak calling hours
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

## Checking Examples of Call Journeys

### Checking for 'No Activity'

In [399]:
# From this list, can be seen that 'No activity' is not an activity name present in the dataset 
df_main['Activity Name'].unique()

array([nan, 'LanguageSelectionMenu', 'MainMenu', 'SeniorsMenu',
       'LegalMenu1', 'LegalMenu2', 'EmploymentMenu', 'ClosedQueueMenu',
       'FamilyMenu', 'OtherLegalMenu', 'OtherLegalOtherMenu',
       'SeniorsConfirmationMenu', 'SuburbsOrCityMenu', 'SeniorsADAPTMenu',
       'HousingMenu', 'PreTenantMenu', 'TenantMenu',
       'DivorceOrParentingMenu', 'ClinicVoicemailTransfer',
       'DisconnectContact', 'SetCallerID', 'BenefitsMenu',
       'FarmworkerMainMenu', 'FrontDeskTransfer1',
       'HelpWithLegalorOtherReasonMenu', 'StaffDirectoryEnglishTransfer',
       'AppointmentMenu', 'FrontDeskTransfer2',
       'ComplimentOrComplaintMenu', 'OtherLegalCriminalCaseMenu',
       'OtherLegalPersonalInjuryMenu', 'ClosedMenu', 'DisconnectContact1',
       'GetLoggedInConsumerAgents', 'IntakePreQueueMessage1',
       'ConsumerQueue', 'PreQueueMessage2', 'PlayMOH300s', 'QueueMenu1',
       'ReadANI', 'CCB', 'PlayCCBConfirmation', 'CallbackRetry',
       'AddressFaxHoursMenu', 'CriminalRe

### Finding the Activity Name Anywhere in the Call Journey

In [400]:
# Filtering the dataframe for rows with a specified value as the acivity name
screen_pop = df_main.loc[df_main['Activity Name'] == 'LegalServerScreenPop','Contact Session ID']

# Finding all the contact session ids for those calls
screen_pop_cals = df_main.loc[df_main['Contact Session ID'].isin(screen_pop), :]

In [401]:
# Unique contact session IDs for those values 
screen_pop_cals['Contact Session ID'].unique()

array(['04617841-2326-4535-bd4c-a9954f1cf4d7',
       '06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45',
       '07f88374-f76b-4b0a-819e-0de67976e69a', ...,
       '289669ce-eab1-4a70-9aee-9eb7362642a2',
       'da2bb927-df57-46c7-9a34-0e973c62fd3f',
       '08c24ff8-835b-4005-b866-29e21116d83f'], dtype=object)

In [402]:
# Example of call journey
    # Final five rows only shown to avoid excessive output 
df_main.loc[df_main['Contact Session ID'] == '06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45', :].tail()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
1251,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,ScreenPopProcessComplete,2025-01-14 13:32:00,NaN,NaN,NaN,13
1252,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,NaN,2025-01-14 13:32:05,SubSenior Other,Flora Fell,NaN,13
1253,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,All LAC Queues Telephony EP,NaN,NaN,2025-01-14 13:39:29,SubSenior Other,Flora Fell,NaN,13
1254,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,NaN,NaN,NaN,2025-01-14 13:39:29,SubSenior Other,Flora Fell,NaN,13
1255,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,NaN,NaN,NaN,2025-01-14 13:39:39,SubSenior Other,Flora Fell,NaN,13


### Finding the Activity Name as Final Activity in Call Journey

In [403]:
# Sorting dataframe by contact session ID and time of each step in call 
sorted_df = df_main.sort_values(["Contact Session ID", "Activity Start Timestamp"])

# Finding rows with non-null activity names
df_nonnull = sorted_df[sorted_df["Activity Name"].notna()]

# Findng the last non-null activity names for each contact session ID 
last_activities = df_nonnull.groupby("Contact Session ID").tail(1)

# Filtering for calls with a specified final activity name 
target = "ScreenPopProcessComplete"
result = last_activities[last_activities["Activity Name"] == target]

# Finding the unique contact session IDs 
session_ids = result["Contact Session ID"].unique()


In [404]:
session_ids

array(['0095d75b-7e36-43bd-90a9-965e3c4722ed',
       '00d7e8e4-863b-41ce-ad69-2609bfe799de',
       '014de6cb-f912-4c78-8e3e-dd5288af51eb',
       '0223c263-3afa-4df6-856f-598d90911f40',
       '02400a1b-c865-4c86-928e-67a0ba56e1f1',
       '025170b2-34d9-4a9d-95a0-51d86f655ef6',
       '02919651-06be-46fa-a421-59ffe7e56976',
       '02ac8ff0-514f-4afe-9649-29d223372fb0',
       '02fdff16-ce55-4761-b5ef-29e128e6efe4',
       '030e4410-1c23-4904-82fe-a1b4befa5eda',
       '033a06b1-e00a-4864-963e-0305aec097ac',
       '03d381c2-5e4a-4b70-ad8d-a07b0e9f1379',
       '03fe93c3-22e5-4bd3-bb0d-339f53ba9012',
       '04a514cd-381f-48ba-b696-57ae0cf43dac',
       '0576f2ec-3db3-4c6b-8f7f-af40cb63f105',
       '05cc2e4b-f749-474e-b975-0c4b0d27319b',
       '05e6b938-c8b9-4b0f-9a45-2a8accd20d94',
       '064b9b80-9e2a-4239-8369-0d8e4652b5b1',
       '0689e1e1-3c0a-4bbe-b5cb-d9181fd86c62',
       '068b691a-edea-4631-8497-9851370b1483',
       '06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45',
       '06efa

In [405]:
# Example of call journey
    # Final five rows only shown to avoid excessive output 
df_main.loc[df_main['Contact Session ID'] == 'ffeba08a-2298-40a2-8864-ecb0a414e7e8', :].tail()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
2456984,ffeba08a-2298-40a2-8864-ecb0a414e7e8,All LAC Queues Telephony EP,NaN,ScreenPopProcessComplete,2025-05-29 12:27:26,NaN,NaN,NaN,12
2456985,ffeba08a-2298-40a2-8864-ecb0a414e7e8,All LAC Queues Telephony EP,NaN,NaN,2025-05-29 12:27:30,SubSenior Benefits,Flora Fell,NaN,12
2457206,ffeba08a-2298-40a2-8864-ecb0a414e7e8,All LAC Queues Telephony EP,NaN,NaN,2025-05-29 12:40:53,SubSenior Benefits,Flora Fell,Customer Left,12
2457207,ffeba08a-2298-40a2-8864-ecb0a414e7e8,NaN,NaN,NaN,2025-05-29 12:40:53,SubSenior Benefits,Flora Fell,NaN,12
2457209,ffeba08a-2298-40a2-8864-ecb0a414e7e8,NaN,NaN,NaN,2025-05-29 12:40:55,SubSenior Benefits,Flora Fell,NaN,12


### 'GetLoggedIn<*>Agents' Examples 

#### There is a termination reason and the NaN activity comes after the final activity name

In [424]:
df_main.loc[df_main['Contact Session ID'] == 'ed3e6cf0-0a99-4e23-8722-4dd064e8e29e', :].tail()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
2359560,ed3e6cf0-0a99-4e23-8722-4dd064e8e29e,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SuburbanSeniorsMenu,2025-05-13 11:17:37,NaN,NaN,NaN,11
2359568,ed3e6cf0-0a99-4e23-8722-4dd064e8e29e,Pre-Legal Menu Seniors Menu Telephony EP,NaN,PreTenantMenu,2025-05-13 11:18:05,NaN,NaN,NaN,11
2359583,ed3e6cf0-0a99-4e23-8722-4dd064e8e29e,NaN,Queues,NaN,2025-05-13 11:18:41,NaN,NaN,NaN,11
2359584,ed3e6cf0-0a99-4e23-8722-4dd064e8e29e,All LAC Queues Telephony EP,NaN,GetLoggedInSubSeniorTenantAgents,2025-05-13 11:18:41,NaN,NaN,NaN,11
2359585,ed3e6cf0-0a99-4e23-8722-4dd064e8e29e,All LAC Queues Telephony EP,NaN,NaN,2025-05-13 11:18:41,NaN,NaN,Customer Left,11


### There is no termination reason and the NaN comes before the final activity name

In [426]:
df_main.loc[df_main['Contact Session ID'] == 'bf1b3761-0c49-47cd-b063-559739afe159', :].tail()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
38480,bf1b3761-0c49-47cd-b063-559739afe159,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SuburbsOrCityMenu,2025-01-15 13:30:45,NaN,NaN,NaN,13
38481,bf1b3761-0c49-47cd-b063-559739afe159,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SuburbanSeniorsMenu,2025-01-15 13:30:45,NaN,NaN,NaN,13
38482,bf1b3761-0c49-47cd-b063-559739afe159,Pre-Legal Menu Seniors Menu Telephony EP,NaN,PreTenantMenu,2025-01-15 13:31:16,NaN,NaN,NaN,13
38483,bf1b3761-0c49-47cd-b063-559739afe159,All LAC Queues Telephony EP,NaN,NaN,2025-01-15 13:31:50,NaN,NaN,NaN,13
38484,bf1b3761-0c49-47cd-b063-559739afe159,All LAC Queues Telephony EP,NaN,GetLoggedInSubSeniorTenantAgents,2025-01-15 13:31:50,NaN,NaN,NaN,13
